In [3]:
# Import libraries
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

# Load dataset
df = pd.read_csv("heart.csv")

# Display basic info
print(df.head())

# Separate features and target
X = df.drop("target", axis=1)   # change if your target column name differs
y = df["target"]

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numerical_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numerical columns:", numerical_cols)

# Label Encoding for binary categorical columns
le = LabelEncoder()
for col in categorical_cols:
    if X[col].nunique() == 2:
        X[col] = le.fit_transform(X[col])

# Update categorical columns after label encoding
categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()

# One-Hot Encoding + Scaling using ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numerical_cols),
        ("cat", OneHotEncoder(drop="first"), categorical_cols)
    ]
)

# Split data
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Define models
models = {
    "SVM": SVC(),
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Random Forest": RandomForestClassifier()
}

# Train and evaluate models (without PCA)
print("\n--- Without PCA ---")
results = {}

for name, model in models.items():
    pipeline = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("classifier", model)
    ])

    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    results[name] = acc

    print(f"{name} Accuracy: {acc:.4f}")

# Best model before PCA
best_model_name = max(results, key=results.get)
print("\nBest Model (without PCA):", best_model_name)

# Apply PCA
pca_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("pca", PCA(n_components=0.95))  # keep 95% variance
])

# Transform data
X_train_pca = pca_pipeline.fit_transform(X_train)
X_test_pca = pca_pipeline.transform(X_test)

# Train models again with PCA
print("\n--- With PCA ---")
pca_results = {}

for name, model in models.items():
    model.fit(X_train_pca, y_train)
    y_pred = model.predict(X_test_pca)

    acc = accuracy_score(y_test, y_pred)
    pca_results[name] = acc

    print(f"{name} Accuracy with PCA: {acc:.4f}")

# Best model after PCA
best_pca_model = max(pca_results, key=pca_results.get)
print("\nBest Model (with PCA):", best_pca_model)

# Compare results
print("\n--- Comparison ---")
for model in models.keys():
    print(f"{model}: Without PCA = {results[model]:.4f}, With PCA = {pca_results[model]:.4f}")

   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   3       145   233    1        0      150      0      2.3      0   
1   37    1   2       130   250    0        1      187      0      3.5      0   
2   41    0   1       130   204    0        0      172      0      1.4      2   
3   56    1   1       120   236    0        1      178      0      0.8      2   
4   57    0   0       120   354    0        1      163      1      0.6      2   

   ca  thal  target  
0   0     1       1  
1   0     2       1  
2   0     2       1  
3   0     2       1  
4   0     2       1  
Categorical columns: []
Numerical columns: ['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']

--- Without PCA ---
SVM Accuracy: 0.8689
Logistic Regression Accuracy: 0.8525
Random Forest Accuracy: 0.8689

Best Model (without PCA): SVM

--- With PCA ---
SVM Accuracy with PCA: 0.8361
Logistic Regression Accuracy with PC

In [2]:
from google.colab import files
import io

print("Please upload the 'heart.csv' file:")
uploaded = files.upload()

for fn in uploaded.keys():
  print('User uploaded file "{name}" with length {length} bytes'.format(
      name=fn, length=len(uploaded[fn])))


Please upload the 'heart.csv' file:


Saving heart.csv to heart.csv
User uploaded file "heart.csv" with length 11021 bytes
